# 📚 Notebook 2: Segmented Log (the BETTER way)

Instead of one infinite file, we keep a **series of segment files**. When the active segment hits a size threshold, we *roll* to a new one. Old segments can be deleted (or archived) with a single `os.unlink` — no rewriting.

This is how Kafka, BookKeeper, and most LSM-tree storage engines organize their on-disk logs.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/segmented-log
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟨 Implementation

In [1]:
import os, tempfile, glob, time

class SegmentedLog:
    """A tiny segmented log. Records are framed as [4-byte big-endian length][payload]."""

    def __init__(self, dir_, max_segment_bytes=4096):
        self.dir = dir_
        os.makedirs(dir_, exist_ok=True)
        self.max_bytes = max_segment_bytes
        # Discover existing segments on restart so we don't overwrite them.
        existing = sorted(glob.glob(os.path.join(dir_, 'segment-*.log')))
        self.active_id = int(existing[-1].split('-')[-1].split('.')[0]) if existing else 0
        self._open_active()

    def _seg_path(self, sid):
        return os.path.join(self.dir, f'segment-{sid:08d}.log')

    def _open_active(self):
        self.fp = open(self._seg_path(self.active_id), 'ab')
        self.active_size = self.fp.tell()  # resume where we left off

    def append(self, record: bytes):
        framed = len(record).to_bytes(4, 'big') + record
        # Roll BEFORE writing so a single record never spans two segments.
        if self.active_size + len(framed) > self.max_bytes and self.active_size > 0:
            self.roll()
        self.fp.write(framed)
        self.active_size += len(framed)

    def roll(self):
        self.fp.close()
        self.active_id += 1
        self._open_active()
        print(f'  rolled to segment {self.active_id}')

    def segments(self):
        return sorted(glob.glob(os.path.join(self.dir, 'segment-*.log')))

    def read_all(self):
        # Reading is just: open each segment in order, decode framed records.
        # flush() pushes Python buffers into the OS so readers see the latest bytes.
        self.fp.flush()
        out = []
        for path in self.segments():
            with open(path, 'rb') as f:
                while True:
                    hdr = f.read(4)
                    if not hdr:
                        break
                    n = int.from_bytes(hdr, 'big')
                    out.append(f.read(n))
        return out

    def delete_segments_before(self, sid):
        for p in self.segments():
            sid_in_path = int(p.split('-')[-1].split('.')[0])
            if sid_in_path < sid:
                os.unlink(p)
                print(f'  removed {os.path.basename(p)}')

    def close(self):
        self.fp.close()

WORKDIR = tempfile.mkdtemp(prefix='seglog_')
log = SegmentedLog(WORKDIR, max_segment_bytes=512)
for i in range(200):
    log.append(f'event-{i:04d}'.encode())

print()
print('segments on disk:')
for p in log.segments():
    print(' ', os.path.basename(p), os.path.getsize(p), 'bytes')


  rolled to segment 1
  rolled to segment 2
  rolled to segment 3
  rolled to segment 4
  rolled to segment 5

segments on disk:
  segment-00000000.log 504 bytes
  segment-00000001.log 504 bytes
  segment-00000002.log 504 bytes
  segment-00000003.log 504 bytes
  segment-00000004.log 504 bytes
  segment-00000005.log 0 bytes


## Reading: the log still works like a log

The segmentation is an **on-disk detail**. Consumers should see a single continuous stream. Our `read_all` just walks the segments in order.

In [2]:
records = log.read_all()
print(f'total records read back: {len(records)}')
print('first 3:', records[:3])
print('last 3: ', records[-3:])
assert records[0] == b'event-0000' and records[-1] == b'event-0199'
print('OK: contents match what we appended, across all segments')

total records read back: 200
first 3: [b'event-0000', b'event-0001', b'event-0002']
last 3:  [b'event-0197', b'event-0198', b'event-0199']
OK: contents match what we appended, across all segments


## 🧹 Compaction / retention by segment

Dropping old data is now an `unlink` — **O(1) per segment**, regardless of how many records are inside.

In [3]:
t0 = time.perf_counter()
log.delete_segments_before(log.active_id - 2)  # keep only the last 2 segments
print(f'compaction done in {time.perf_counter()-t0:.6f}s')
print('segments left:', len(log.segments()))


  removed segment-00000000.log
  removed segment-00000001.log
  removed segment-00000002.log
compaction done in 0.001226s
segments left: 3


## 📊 Compare: single file vs segmented

| Concern | Single file | Segmented |
|---|---|---|
| Drop oldest data | rewrite whole file (O(n)) | unlink files (O(segments)) |
| Parallel reads | one fd contention | one fd per segment |
| Backup/ship | huge atomic file | incremental segments |
| Crash recovery scan | scan everything | scan only the active segment |

## 🚀 Best practices

- Use **time-based** *or* **size-based** rolling (whichever happens first).
- Keep a small **sparse index** per segment (offset → file position) so you can binary-search rather than scan.
- Make segment names sortable (zero-padded numeric IDs) so directory listing == log order.

## Where you see this in the real world

- **Apache Kafka** - each partition is a directory of `.log` segment files plus `.index` files. Retention is "delete segments older than 7 days" = `os.unlink`.
- **PostgreSQL WAL** - 16 MiB segments in `pg_wal/`. Old ones get recycled or archived.
- **etcd / Raft WAL** - fixed-size segments; snapshots let the cluster drop old segments.
- **LevelDB / RocksDB / Cassandra** - SSTables are an immutable variant of this idea: write a new file, delete old ones during compaction.

The pattern is everywhere because the alternative (one unbounded file) simply doesn't scale.

Next up: [Notebook 3](./03_sparse_index_and_recovery.ipynb) adds a **sparse index** for fast offset lookup and shows **crash recovery** when a write is cut in half.